In [52]:
# ===============================
# 📦 Bibliotecas principais
# ===============================

# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Modelagem e machine learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Engenharia de features
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Data & time
import datetime as dt
import time
from dateutil import parser

# Sistema e logging
import os
import logging
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Utilitários gerais
import json
import joblib
import gc




In [70]:
%pip install imblearn

from imblearn.over_sampling import SMOTE









  Using cached imblearn-0.0-py2.py3-none-any.whl.metadata (355 bytes)
  Using cached imbalanced_learn-0.13.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached sklearn_compat-0.1.3-py3-none-any.whl.metadata (18 kB)
Using cached imblearn-0.0-py2.py3-none-any.whl (1.9 kB)
Using cached imbalanced_learn-0.13.0-py3-none-any.whl (238 kB)
Using cached sklearn_compat-0.1.3-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [53]:
RAW_DIR = Path(r"C:\Users\DoIt\Downloads\data-science-challenge-at-eef-2024\spaceflights\data\01_raw")

# Caminhos completos dos arquivos
public_path = RAW_DIR / "public.csv"
sample_path = RAW_DIR / "sample.csv"
features_path = RAW_DIR / "features_pca_95.parquet"


In [54]:
df_sample = pd.read_csv("C:/Users/Doit/Desktop/data-science-challenge-at-eef-2024/spaceflights/data/01_raw/sample.csv")
df_public = pd.read_csv("C:/Users/Doit/Desktop/data-science-challenge-at-eef-2024/spaceflights/data/01_raw/public.csv")

In [55]:
df_public.head()

,flightid,hora_ref,origem,destino,url_img_satelite,metaf,metar,prev_troca_cabeceira,troca_cabeceira_hora_anterior,espera
0,504a62621cd231d6ab67e674ce538cd3,2022-06-01T01:00:00Z,SBCF,SBFL,http://satelite.cptec.inpe.br/repositoriogoes/...,NaN,METAR SBFL 010000Z 17009KT 140V200 9999 BKN030...,0,1,0.0
1,b0fd0f83644625ecc21f5261e8e5e347,2022-06-01T01:00:00Z,SBPA,SBFL,http://satelite.cptec.inpe.br/repositoriogoes/...,NaN,METAR SBFL 010000Z 17009KT 140V200 9999 BKN030...,0,1,0.0
2,1210f0ca07ddca00d09a3e02d3b100d8,2022-06-01T01:00:00Z,SBSP,SBCF,http://satelite.cptec.inpe.br/repositoriogoes/...,NaN,METAR SBCF 010000Z 12006KT CAVOK 21/14 Q1018=,0,0,0.0
3,b25032f34507cce285ee779446496568,2022-06-01T01:00:00Z,SBGR,SBCT,http://satelite.cptec.inpe.br/repositoriogoes/...,NaN,METAR SBCT 010000Z 10006KT 7000 -RA BKN004 OVC...,0,0,0.0
4,00762a9892ecba7c66d1d87800d38cac,2022-06-01T01:00:00Z,SBSP,SBSV,http://satelite.cptec.inpe.br/repositoriogoes/...,NaN,METAR SBSV 010000Z 11008KT 9999 FEW023 27/21 Q...,0,1,0.0


In [56]:
df_public['metaf'].fillna('DESCONHECIDO', inplace=True)
df_public['hora_ref'] = pd.to_datetime(df_public['hora_ref'])
df_public.duplicated().sum()
df_public.drop_duplicates(inplace=True)
df_public['url_img_satelite'].isnull().sum()



np.int64(3733)

In [57]:
df_public['hora'] = df_public['hora_ref'].dt.hour
df_public['dia'] = df_public['hora_ref'].dt.date
df_public['precisa_troca'] = df_public['prev_troca_cabeceira'] | df_public['troca_cabeceira_hora_anterior']


In [58]:
df_public = pd.get_dummies(df_public, columns=['origem', 'destino'])


In [59]:
df_public['espera'].describe()


count    211679.000000
mean          0.017413
std           0.130805
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: espera, dtype: float64

In [60]:
df_public['espera'].value_counts(normalize=True)


espera
0.0    0.982587
1.0    0.017413
Name: proportion, dtype: float64

In [61]:
df_public.columns

Index(['flightid', 'hora_ref', 'url_img_satelite', 'metaf', 'metar',
       'prev_troca_cabeceira', 'troca_cabeceira_hora_anterior', 'espera',
       'hora', 'dia', 'precisa_troca', 'origem_SBBR', 'origem_SBCF',
       'origem_SBCT', 'origem_SBFL', 'origem_SBGL', 'origem_SBGR',
       'origem_SBKP', 'origem_SBPA', 'origem_SBRF', 'origem_SBRJ',
       'origem_SBSP', 'origem_SBSV', 'destino_SBBR', 'destino_SBCF',
       'destino_SBCT', 'destino_SBFL', 'destino_SBGL', 'destino_SBGR',
       'destino_SBKP', 'destino_SBPA', 'destino_SBRF', 'destino_SBRJ',
       'destino_SBSP', 'destino_SBSV'],
      dtype='object')

In [62]:
# Dados de treino = onde espera não é NaN
df_train = df_public[df_public['espera'].notnull()]

# Dados a prever = onde espera é NaN
df_pred = df_public[df_public['espera'].isnull()]


In [63]:
X_to_predict = df_pred.drop(columns=['espera', 'flightid', 'url_img_satelite', 'metar', 'metaf'])


In [78]:
# 1) Dataset de treino e previsão
X_train = df_train.drop(columns=['espera', 'flightid', 'url_img_satelite', 'metar', 'metaf'])
y_train = df_train['espera']

X_to_predict = df_pred.drop(columns=['espera', 'flightid', 'url_img_satelite', 'metar', 'metaf'])

# 2) Só alinhar as colunas pra garantir match:
X_train, X_to_predict = X_train.align(X_to_predict, join='left', axis=1, fill_value=0)


In [79]:
print(X_tr.dtypes)


prev_troca_cabeceira             int64
troca_cabeceira_hora_anterior    int64
hora                             int32
dia                              int32
precisa_troca                    int64
origem_SBBR                       bool
origem_SBCF                       bool
origem_SBCT                       bool
origem_SBFL                       bool
origem_SBGL                       bool
origem_SBGR                       bool
origem_SBKP                       bool
origem_SBPA                       bool
origem_SBRF                       bool
origem_SBRJ                       bool
origem_SBSP                       bool
origem_SBSV                       bool
destino_SBBR                      bool
destino_SBCF                      bool
destino_SBCT                      bool
destino_SBFL                      bool
destino_SBGL                      bool
destino_SBGR                      bool
destino_SBKP                      bool
destino_SBPA                      bool
destino_SBRF             

In [80]:
X_train['dia'] = pd.to_datetime(X_train['dia']).dt.dayofyear
X_to_predict['dia'] = pd.to_datetime(X_to_predict['dia']).dt.dayofyear


In [81]:
X_train = X_train.drop(columns=['hora_ref'], errors='ignore')
X_to_predict = X_to_predict.drop(columns=['hora_ref'], errors='ignore')


In [82]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

sm = SMOTE(random_state=42)
X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)


In [84]:
clf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

# Treine
clf.fit(X_tr_res, y_tr_res)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [85]:
# Agora prever os casos NaN
espera_pred = clf.predict(X_to_predict)

# Inserir no df original
df_pred['espera_pred'] = espera_pred

# Checar distribuição
print(df_pred['espera_pred'].value_counts())


espera_pred
0.0    89721
1.0      999
Name: count, dtype: int64


In [88]:
# Prever os casos NaN
espera_pred = clf.predict(X_to_predict)

In [89]:
print(df_pred.shape)
print(df_pred.columns)


(90720, 36)
Index(['flightid', 'hora_ref', 'url_img_satelite', 'metaf', 'metar',
       'prev_troca_cabeceira', 'troca_cabeceira_hora_anterior', 'espera',
       'hora', 'dia', 'precisa_troca', 'origem_SBBR', 'origem_SBCF',
       'origem_SBCT', 'origem_SBFL', 'origem_SBGL', 'origem_SBGR',
       'origem_SBKP', 'origem_SBPA', 'origem_SBRF', 'origem_SBRJ',
       'origem_SBSP', 'origem_SBSV', 'destino_SBBR', 'destino_SBCF',
       'destino_SBCT', 'destino_SBFL', 'destino_SBGL', 'destino_SBGR',
       'destino_SBKP', 'destino_SBPA', 'destino_SBRF', 'destino_SBRJ',
       'destino_SBSP', 'destino_SBSV', 'espera_pred'],
      dtype='object')


In [90]:
df_submit = df_pred[['flightid', 'espera_pred']]


In [91]:
print(df_submit.shape)
print(df_submit.head())


(90720, 2)
                                flightid  espera_pred
211679  45e7978b9d88f934cc06c11b6f0edba7          0.0
211680  16ed22b3755aa9196d16fdd2a173c98f          0.0
211681  b548d2c700496e2536d78caf626aee17          0.0
211682  e4cc2545104bcfe978912d39f0960f4e          0.0
211683  ace87fdae884359186e9851c38b146fb          0.0


In [92]:
df_submit.to_csv('submission.csv', index=False)
